# Naive Bayes Classification - From Scratch Implementation

## Table of Contents
1. [Theory & Mathematical Foundation](#theory)
2. [Implementation from Scratch](#implementation)
3. [Training & Optimization](#training)
4. [Diagnostics & Evaluation](#diagnostics)
5. [Visualizations](#visualizations)
6. [Use Cases & Guidelines](#use-cases)
7. [Comparison with sklearn](#comparison)

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_iris, load_wine, fetch_20newsgroups
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_recall_fscore_support, roc_curve, auc
)
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Theory & Mathematical Foundation <a id='theory'></a>

### What is Naive Bayes?

Naive Bayes is a family of **probabilistic classifiers** based on Bayes' theorem with a strong (naive) assumption of **conditional independence** between features given the class label.

---

### Bayes' Theorem

The foundation of Naive Bayes is **Bayes' Theorem**:

$$P(C|X) = \frac{P(X|C) \cdot P(C)}{P(X)}$$

Where:
- **$P(C|X)$** = **Posterior probability**: Probability of class $C$ given features $X$
- **$P(X|C)$** = **Likelihood**: Probability of features $X$ given class $C$
- **$P(C)$** = **Prior probability**: Probability of class $C$ before observing data
- **$P(X)$** = **Evidence**: Probability of features $X$ (normalizing constant)

---

### The Naive Assumption

The "naive" assumption is that all features are **conditionally independent** given the class:

$$P(X|C) = P(x_1, x_2, ..., x_n|C) = \prod_{i=1}^{n} P(x_i|C)$$

This simplifies the computation dramatically!

---

### Classification Rule

Since $P(X)$ is constant for all classes, we classify using:

$$\hat{y} = \arg\max_{c} P(C=c) \prod_{i=1}^{n} P(x_i|C=c)$$

In practice, we use **log probabilities** to avoid numerical underflow:

$$\hat{y} = \arg\max_{c} \left[ \log P(C=c) + \sum_{i=1}^{n} \log P(x_i|C=c) \right]$$

---

### Types of Naive Bayes Classifiers

#### 1. Gaussian Naive Bayes
Assumes features follow a **normal distribution**:

$$P(x_i|C=c) = \frac{1}{\sqrt{2\pi\sigma_c^2}} \exp\left(-\frac{(x_i - \mu_c)^2}{2\sigma_c^2}\right)$$

**Use case**: Continuous features (e.g., measurements, sensor data)

#### 2. Multinomial Naive Bayes
Assumes features represent **counts or frequencies**:

$$P(x_i|C=c) = \frac{N_{ci} + \alpha}{N_c + \alpha n}$$

Where:
- $N_{ci}$ = count of feature $i$ in class $c$
- $N_c$ = total count of all features in class $c$
- $\alpha$ = smoothing parameter (Laplace smoothing)
- $n$ = number of features

**Use case**: Text classification (word counts, TF-IDF)

#### 3. Bernoulli Naive Bayes
Assumes features are **binary** (present/absent):

$$P(x_i|C=c) = P(x_i=1|C=c)^{x_i} \cdot (1 - P(x_i=1|C=c))^{(1-x_i)}$$

**Use case**: Binary features (e.g., word presence/absence, binary attributes)

---

### Time & Space Complexity

| Operation | Time Complexity | Space Complexity |
|-----------|-----------------|------------------|
| Training (Gaussian) | O(n_samples * n_features) | O(n_classes * n_features) |
| Training (Multinomial) | O(n_samples * n_features) | O(n_classes * n_features) |
| Training (Bernoulli) | O(n_samples * n_features) | O(n_classes * n_features) |
| Prediction | O(n_classes * n_features) | O(1) |

**Key advantage**: Linear complexity makes Naive Bayes very efficient for large datasets!

---

### Key Assumptions & Properties

1. **Feature Independence**: Features are independent given the class (often violated but still works!)
2. **Generative Model**: Models the joint probability $P(X, C)$
3. **No Iterative Training**: Parameters computed directly from data (no gradient descent)
4. **Probability Calibration**: Probabilities may not be well-calibrated but rankings are good

## 2. Implementation from Scratch <a id='implementation'></a>

In [ ]:
class GaussianNaiveBayesScratch:
    """
    Gaussian Naive Bayes classifier implemented from scratch.
    
    Assumes features follow a Gaussian (normal) distribution.
    Best suited for continuous features.
    
    Parameters:
    -----------
    var_smoothing : float, default=1e-9
        Portion of the largest variance of all features added to variances
        for numerical stability.
    """
    
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing
        self.classes_ = None
        self.class_prior_ = None
        self.theta_ = None  # Mean of each feature per class
        self.var_ = None    # Variance of each feature per class
        self.epsilon_ = None  # Smoothing term
    
    def fit(self, X, y):
        """
        Fit Gaussian Naive Bayes according to X, y.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training vectors
        y : array-like of shape (n_samples,)
            Target values
        
        Returns:
        --------
        self : object
        """
        X = np.array(X)
        y = np.array(y)
        
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        # Initialize arrays for parameters
        self.theta_ = np.zeros((n_classes, n_features))
        self.var_ = np.zeros((n_classes, n_features))
        self.class_prior_ = np.zeros(n_classes)
        
        # Calculate parameters for each class
        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            
            # Prior probability P(C)
            self.class_prior_[idx] = X_c.shape[0] / n_samples
            
            # Mean of each feature for this class
            self.theta_[idx, :] = X_c.mean(axis=0)
            
            # Variance of each feature for this class
            self.var_[idx, :] = X_c.var(axis=0)
        
        # Add smoothing term for numerical stability
        self.epsilon_ = self.var_smoothing * np.max(self.var_)
        self.var_ += self.epsilon_
        
        return self
    
    def _compute_log_likelihood(self, X):
        """
        Compute log-likelihood of samples for each class.
        
        Uses the Gaussian PDF:
        log P(x|C) = -0.5 * [log(2*pi*var) + (x - mean)^2 / var]
        """
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        log_likelihood = np.zeros((n_samples, n_classes))
        
        for idx in range(n_classes):
            # Log of Gaussian PDF
            log_var = np.log(2 * np.pi * self.var_[idx, :])
            diff = X - self.theta_[idx, :]
            log_prob = -0.5 * np.sum(log_var + (diff ** 2) / self.var_[idx, :], axis=1)
            log_likelihood[:, idx] = log_prob
        
        return log_likelihood
    
    def _compute_log_posterior(self, X):
        """
        Compute log-posterior probability: log P(C) + log P(X|C)
        """
        log_prior = np.log(self.class_prior_)
        log_likelihood = self._compute_log_likelihood(X)
        return log_likelihood + log_prior
    
    def predict(self, X):
        """
        Perform classification on samples in X.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
        
        Returns:
        --------
        y_pred : ndarray of shape (n_samples,)
            Predicted class labels
        """
        X = np.array(X)
        log_posterior = self._compute_log_posterior(X)
        return self.classes_[np.argmax(log_posterior, axis=1)]
    
    def predict_proba(self, X):
        """
        Return probability estimates for the test data X.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
        
        Returns:
        --------
        proba : ndarray of shape (n_samples, n_classes)
            Probability of each class for each sample
        """
        X = np.array(X)
        log_posterior = self._compute_log_posterior(X)
        
        # Convert log probabilities to probabilities using softmax
        # Subtract max for numerical stability
        log_posterior -= np.max(log_posterior, axis=1, keepdims=True)
        proba = np.exp(log_posterior)
        proba /= np.sum(proba, axis=1, keepdims=True)
        
        return proba
    
    def score(self, X, y):
        """
        Return the mean accuracy on the given test data and labels.
        """
        return np.mean(self.predict(X) == y)

In [ ]:
class MultinomialNaiveBayesScratch:
    """
    Multinomial Naive Bayes classifier implemented from scratch.
    
    Suitable for discrete features representing counts or frequencies.
    Commonly used for text classification with word counts or TF-IDF.
    
    Parameters:
    -----------
    alpha : float, default=1.0
        Additive (Laplace/Lidstone) smoothing parameter.
        alpha=0 means no smoothing.
        alpha=1 is Laplace smoothing.
    """
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_prior_ = None
        self.feature_log_prob_ = None  # Log probability of features per class
        self.class_count_ = None
        self.feature_count_ = None
    
    def fit(self, X, y):
        """
        Fit Multinomial Naive Bayes according to X, y.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training vectors (should be non-negative)
        y : array-like of shape (n_samples,)
            Target values
        
        Returns:
        --------
        self : object
        """
        X = np.array(X)
        y = np.array(y)
        
        # Check for non-negative values
        if np.any(X < 0):
            raise ValueError("Multinomial NB requires non-negative features.")
        
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        # Initialize arrays
        self.class_count_ = np.zeros(n_classes)
        self.feature_count_ = np.zeros((n_classes, n_features))
        
        # Count features for each class
        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.class_count_[idx] = X_c.shape[0]
            # Sum of feature counts for each class
            self.feature_count_[idx, :] = X_c.sum(axis=0)
        
        # Compute class priors
        self.class_prior_ = self.class_count_ / n_samples
        
        # Compute feature log probabilities with smoothing
        # P(x_i|C) = (count(x_i, C) + alpha) / (sum(count(x, C)) + alpha * n_features)
        smoothed_fc = self.feature_count_ + self.alpha
        smoothed_cc = smoothed_fc.sum(axis=1, keepdims=True)
        self.feature_log_prob_ = np.log(smoothed_fc / smoothed_cc)
        
        return self
    
    def _compute_log_posterior(self, X):
        """
        Compute log-posterior: log P(C) + sum(x_i * log P(x_i|C))
        """
        log_prior = np.log(self.class_prior_)
        # For multinomial, likelihood is: product of P(x_i|C)^x_i
        # In log space: sum of x_i * log P(x_i|C)
        log_likelihood = X @ self.feature_log_prob_.T
        return log_likelihood + log_prior
    
    def predict(self, X):
        """
        Perform classification on samples in X.
        """
        X = np.array(X)
        log_posterior = self._compute_log_posterior(X)
        return self.classes_[np.argmax(log_posterior, axis=1)]
    
    def predict_proba(self, X):
        """
        Return probability estimates for the test data X.
        """
        X = np.array(X)
        log_posterior = self._compute_log_posterior(X)
        
        # Softmax to convert to probabilities
        log_posterior -= np.max(log_posterior, axis=1, keepdims=True)
        proba = np.exp(log_posterior)
        proba /= np.sum(proba, axis=1, keepdims=True)
        
        return proba
    
    def score(self, X, y):
        """
        Return the mean accuracy on the given test data and labels.
        """
        return np.mean(self.predict(X) == y)

In [ ]:
class BernoulliNaiveBayesScratch:
    """
    Bernoulli Naive Bayes classifier implemented from scratch.
    
    Designed for binary/boolean features. Can handle non-binary input
    by binarizing with the specified threshold.
    
    Parameters:
    -----------
    alpha : float, default=1.0
        Additive (Laplace/Lidstone) smoothing parameter.
    binarize : float or None, default=0.0
        Threshold for binarizing features. If None, input is assumed
        to be already binary.
    """
    
    def __init__(self, alpha=1.0, binarize=0.0):
        self.alpha = alpha
        self.binarize = binarize
        self.classes_ = None
        self.class_prior_ = None
        self.feature_prob_ = None  # P(x_i=1|C) for each feature and class
        self.feature_log_prob_ = None  # Log of above
        self.class_count_ = None
    
    def _binarize_features(self, X):
        """
        Binarize features based on threshold.
        """
        if self.binarize is not None:
            return (X > self.binarize).astype(np.float64)
        return X
    
    def fit(self, X, y):
        """
        Fit Bernoulli Naive Bayes according to X, y.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training vectors
        y : array-like of shape (n_samples,)
            Target values
        
        Returns:
        --------
        self : object
        """
        X = np.array(X)
        y = np.array(y)
        
        # Binarize input
        X = self._binarize_features(X)
        
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        # Initialize arrays
        self.class_count_ = np.zeros(n_classes)
        self.feature_count_ = np.zeros((n_classes, n_features))
        
        # Count features for each class
        for idx, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.class_count_[idx] = X_c.shape[0]
            # Count of feature=1 for each class
            self.feature_count_[idx, :] = X_c.sum(axis=0)
        
        # Compute class priors
        self.class_prior_ = self.class_count_ / n_samples
        
        # Compute feature probabilities with Laplace smoothing
        # P(x_i=1|C) = (count(x_i=1, C) + alpha) / (count(C) + 2*alpha)
        smoothed_count = self.feature_count_ + self.alpha
        smoothed_total = self.class_count_.reshape(-1, 1) + 2 * self.alpha
        self.feature_prob_ = smoothed_count / smoothed_total
        
        # Store log probabilities for numerical stability
        self.feature_log_prob_ = np.log(self.feature_prob_)
        self.neg_feature_log_prob_ = np.log(1 - self.feature_prob_)
        
        return self
    
    def _compute_log_posterior(self, X):
        """
        Compute log-posterior for Bernoulli NB.
        
        For Bernoulli: P(x|C) = prod_i P(x_i=1|C)^x_i * P(x_i=0|C)^(1-x_i)
        
        In log space:
        log P(x|C) = sum_i [x_i * log P(x_i=1|C) + (1-x_i) * log P(x_i=0|C)]
                   = sum_i [x_i * log P(x_i=1|C) + (1-x_i) * log(1-P(x_i=1|C))]
        """
        log_prior = np.log(self.class_prior_)
        
        # Binarize input
        X = self._binarize_features(X)
        
        # Compute log likelihood
        # log P(x|C) = sum_i [x_i * log(p_i) + (1-x_i) * log(1-p_i)]
        # = sum_i [x_i * (log(p_i) - log(1-p_i)) + log(1-p_i)]
        # = X @ (log(p) - log(1-p)).T + sum(log(1-p))
        
        log_odds = self.feature_log_prob_ - self.neg_feature_log_prob_
        log_likelihood = X @ log_odds.T + np.sum(self.neg_feature_log_prob_, axis=1)
        
        return log_likelihood + log_prior
    
    def predict(self, X):
        """
        Perform classification on samples in X.
        """
        X = np.array(X)
        log_posterior = self._compute_log_posterior(X)
        return self.classes_[np.argmax(log_posterior, axis=1)]
    
    def predict_proba(self, X):
        """
        Return probability estimates for the test data X.
        """
        X = np.array(X)
        log_posterior = self._compute_log_posterior(X)
        
        # Softmax to convert to probabilities
        log_posterior -= np.max(log_posterior, axis=1, keepdims=True)
        proba = np.exp(log_posterior)
        proba /= np.sum(proba, axis=1, keepdims=True)
        
        return proba
    
    def score(self, X, y):
        """
        Return the mean accuracy on the given test data and labels.
        """
        return np.mean(self.predict(X) == y)

## 3. Training & Optimization <a id='training'></a>

### Training Gaussian Naive Bayes on Iris Dataset

In [ ]:
# Load the Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("Iris Dataset Summary")
print("=" * 50)
print(f"Number of samples: {X_iris.shape[0]}")
print(f"Number of features: {X_iris.shape[1]}")
print(f"Feature names: {feature_names}")
print(f"Target classes: {target_names}")
print(f"Class distribution: {np.bincount(y_iris)}")

# Display sample statistics
print("\nFeature Statistics:")
df_stats = pd.DataFrame(X_iris, columns=feature_names)
df_stats['target'] = y_iris
print(df_stats.groupby('target').agg(['mean', 'std']).round(2))

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Training class distribution: {np.bincount(y_train)}")
print(f"Test class distribution: {np.bincount(y_test)}")

In [ ]:
# Train Gaussian Naive Bayes
gnb_scratch = GaussianNaiveBayesScratch(var_smoothing=1e-9)
gnb_scratch.fit(X_train, y_train)

print("Gaussian Naive Bayes - Training Complete")
print("=" * 50)
print("\nLearned Parameters:")
print(f"\nClass Priors P(C):")
for i, (prior, name) in enumerate(zip(gnb_scratch.class_prior_, target_names)):
    print(f"  P({name}) = {prior:.4f}")

print(f"\nFeature Means (theta) per class:")
theta_df = pd.DataFrame(
    gnb_scratch.theta_,
    index=target_names,
    columns=feature_names
)
print(theta_df.round(3))

print(f"\nFeature Variances per class:")
var_df = pd.DataFrame(
    gnb_scratch.var_,
    index=target_names,
    columns=feature_names
)
print(var_df.round(3))

In [ ]:
# Evaluate on train and test sets
train_acc = gnb_scratch.score(X_train, y_train)
test_acc = gnb_scratch.score(X_test, y_test)

print("Model Performance:")
print(f"  Training Accuracy: {train_acc:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")

# Make predictions
y_pred = gnb_scratch.predict(X_test)
y_proba = gnb_scratch.predict_proba(X_test)

print("\nSample Predictions (first 5 test samples):")
for i in range(5):
    print(f"  Sample {i+1}: True={target_names[y_test[i]]}, "
          f"Pred={target_names[y_pred[i]]}, "
          f"Probabilities={y_proba[i].round(3)}")

### Training Multinomial Naive Bayes (Text Classification Example)

In [ ]:
# Create a simple text classification dataset
# We'll use a subset of 20 newsgroups for demonstration
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics']
newsgroups = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print("20 Newsgroups Dataset (subset)")
print("=" * 50)
print(f"Categories: {categories}")
print(f"Number of documents: {len(newsgroups.data)}")
print(f"Class distribution: {np.bincount(newsgroups.target)}")

# Show sample document
print("\nSample document:")
print("-" * 50)
print(newsgroups.data[0][:500] + "...")

In [ ]:
# Vectorize text using Count Vectorizer
vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X_text = vectorizer.fit_transform(newsgroups.data).toarray()
y_text = newsgroups.target

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Document-term matrix shape: {X_text.shape}")

# Split data
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    X_text, y_text, test_size=0.3, random_state=42, stratify=y_text
)

print(f"\nTraining set: {X_train_text.shape[0]} documents")
print(f"Test set: {X_test_text.shape[0]} documents")

In [ ]:
# Train Multinomial Naive Bayes with different alpha values
alphas = [0.01, 0.1, 0.5, 1.0, 2.0]
results = []

print("Multinomial NB - Effect of Laplace Smoothing (alpha)")
print("=" * 60)
print(f"{'Alpha':<10} {'Train Acc':<12} {'Test Acc':<12}")
print("-" * 34)

for alpha in alphas:
    mnb = MultinomialNaiveBayesScratch(alpha=alpha)
    mnb.fit(X_train_text, y_train_text)
    train_acc = mnb.score(X_train_text, y_train_text)
    test_acc = mnb.score(X_test_text, y_test_text)
    results.append({'alpha': alpha, 'train_acc': train_acc, 'test_acc': test_acc})
    print(f"{alpha:<10} {train_acc:<12.4f} {test_acc:<12.4f}")

# Use best alpha for further analysis
best_alpha = max(results, key=lambda x: x['test_acc'])['alpha']
print(f"\nBest alpha based on test accuracy: {best_alpha}")

mnb_scratch = MultinomialNaiveBayesScratch(alpha=best_alpha)
mnb_scratch.fit(X_train_text, y_train_text)

In [ ]:
# Visualize the effect of alpha
results_df = pd.DataFrame(results)

plt.figure(figsize=(10, 6))
plt.plot(results_df['alpha'], results_df['train_acc'], 'bo-', label='Training Accuracy', linewidth=2)
plt.plot(results_df['alpha'], results_df['test_acc'], 'ro-', label='Test Accuracy', linewidth=2)
plt.xlabel('Alpha (Smoothing Parameter)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Effect of Laplace Smoothing on Multinomial Naive Bayes', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.tight_layout()
plt.show()

### Training Bernoulli Naive Bayes

In [ ]:
# Train Bernoulli Naive Bayes on binarized text data
# Convert word counts to binary (word present/absent)

bnb_scratch = BernoulliNaiveBayesScratch(alpha=1.0, binarize=0.0)
bnb_scratch.fit(X_train_text, y_train_text)

print("Bernoulli Naive Bayes - Training Complete")
print("=" * 50)

train_acc = bnb_scratch.score(X_train_text, y_train_text)
test_acc = bnb_scratch.score(X_test_text, y_test_text)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Show learned feature probabilities for each class
print("\nTop 10 words with highest P(word|class) for each category:")
vocab = vectorizer.get_feature_names_out()

for idx, category in enumerate(categories):
    top_indices = np.argsort(bnb_scratch.feature_prob_[idx])[-10:][::-1]
    top_words = [vocab[i] for i in top_indices]
    top_probs = bnb_scratch.feature_prob_[idx, top_indices]
    print(f"\n{category}:")
    for word, prob in zip(top_words, top_probs):
        print(f"  {word}: {prob:.4f}")

## 4. Diagnostics & Evaluation <a id='diagnostics'></a>

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion Matrix"):
    """
    Plot a confusion matrix with proper labeling.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title(title, fontsize=14)
    plt.tight_layout()
    plt.show()
    
    return cm

# Confusion matrix for Gaussian NB on Iris
y_pred_iris = gnb_scratch.predict(X_test)
cm_iris = plot_confusion_matrix(
    y_test, y_pred_iris, target_names,
    title="Gaussian Naive Bayes - Iris Dataset"
)

print("\nClassification Report - Gaussian NB (Iris):")
print(classification_report(y_test, y_pred_iris, target_names=target_names))

In [ ]:
# Confusion matrix for Multinomial NB on Text
y_pred_text = mnb_scratch.predict(X_test_text)
cm_text = plot_confusion_matrix(
    y_test_text, y_pred_text, categories,
    title="Multinomial Naive Bayes - 20 Newsgroups"
)

print("\nClassification Report - Multinomial NB (Text):")
print(classification_report(y_test_text, y_pred_text, target_names=categories))

In [ ]:
def plot_learning_curves_nb(X, y, title="Learning Curves"):
    """
    Plot learning curves for Naive Bayes to diagnose bias/variance.
    """
    train_sizes = np.linspace(0.1, 1.0, 10)
    train_scores = []
    val_scores = []
    
    for size in train_sizes:
        n_samples = int(len(X) * size)
        X_subset = X[:n_samples]
        y_subset = y[:n_samples]
        
        if len(np.unique(y_subset)) < 2:
            continue
        
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_subset, y_subset, test_size=0.2, random_state=42
        )
        
        model = GaussianNaiveBayesScratch()
        model.fit(X_tr, y_tr)
        
        train_scores.append(model.score(X_tr, y_tr))
        val_scores.append(model.score(X_val, y_val))
    
    plt.figure(figsize=(10, 6))
    plt.plot(train_sizes[:len(train_scores)] * len(X), train_scores, 
             'o-', label='Training Score', linewidth=2)
    plt.plot(train_sizes[:len(val_scores)] * len(X), val_scores, 
             'o-', label='Validation Score', linewidth=2)
    plt.xlabel('Training Set Size', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot learning curves for Gaussian NB on Iris
plot_learning_curves_nb(X_iris, y_iris, 
                        title="Learning Curves - Gaussian Naive Bayes (Iris)")

In [ ]:
def plot_multiclass_roc(y_true, y_proba, class_names):
    """
    Plot ROC curves for multiclass classification.
    """
    n_classes = len(class_names)
    
    # Binarize labels for ROC
    from sklearn.preprocessing import label_binarize
    y_bin = label_binarize(y_true, classes=range(n_classes))
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = plt.cm.Set1(np.linspace(0, 1, n_classes))
    
    for i, (color, name) in enumerate(zip(colors, class_names)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{name} (AUC = {roc_auc:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves - Multiclass Classification', fontsize=14)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot ROC curves for Gaussian NB on Iris
y_proba_iris = gnb_scratch.predict_proba(X_test)
plot_multiclass_roc(y_test, y_proba_iris, target_names)

In [ ]:
def plot_calibration_curve(y_true, y_proba, n_bins=10, title="Calibration Curve"):
    """
    Plot calibration curve to assess probability calibration.
    """
    # For binary or first class in multiclass
    if y_proba.ndim > 1:
        # Convert to binary problem (class 0 vs rest)
        y_bin = (y_true == 0).astype(int)
        proba = y_proba[:, 0]
    else:
        y_bin = y_true
        proba = y_proba
    
    # Calculate calibration curve
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    true_probs = []
    pred_probs = []
    
    for i in range(n_bins):
        mask = (proba >= bins[i]) & (proba < bins[i+1])
        if mask.sum() > 0:
            true_probs.append(y_bin[mask].mean())
            pred_probs.append(proba[mask].mean())
    
    plt.figure(figsize=(8, 8))
    plt.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
    plt.plot(pred_probs, true_probs, 's-', markersize=8, linewidth=2,
             label='Model Calibration')
    plt.xlabel('Mean Predicted Probability', fontsize=12)
    plt.ylabel('Fraction of Positives', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_calibration_curve(y_test, y_proba_iris, 
                       title="Probability Calibration - Gaussian NB (Iris)")

## 5. Visualizations <a id='visualizations'></a>

In [ ]:
def plot_decision_boundary_2d(model, X, y, feature_indices=(0, 1), 
                               feature_names=None, class_names=None,
                               title="Decision Boundary"):
    """
    Plot decision boundary for 2D projection of data.
    """
    # Extract two features
    X_2d = X[:, feature_indices]
    
    # Fit model on 2D data
    model_2d = GaussianNaiveBayesScratch()
    model_2d.fit(X_2d, y)
    
    # Create mesh grid
    h = 0.02
    x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
    y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Decision regions
    cmap_light = plt.cm.RdYlBu
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_light)
    
    # Decision boundaries
    ax.contour(xx, yy, Z, colors='black', linewidths=1, linestyles='--')
    
    # Data points
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap=plt.cm.RdYlBu,
                         edgecolors='black', s=50)
    
    # Labels
    if feature_names is not None:
        ax.set_xlabel(feature_names[feature_indices[0]], fontsize=12)
        ax.set_ylabel(feature_names[feature_indices[1]], fontsize=12)
    else:
        ax.set_xlabel(f'Feature {feature_indices[0]}', fontsize=12)
        ax.set_ylabel(f'Feature {feature_indices[1]}', fontsize=12)
    
    ax.set_title(title, fontsize=14)
    
    if class_names is not None:
        legend_elements = [plt.scatter([], [], c=plt.cm.RdYlBu(i/2), 
                                       label=name, s=50, edgecolors='black')
                          for i, name in enumerate(class_names)]
        ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
    
    plt.tight_layout()
    plt.show()

# Plot decision boundaries for different feature pairs
feature_pairs = [(0, 1), (2, 3), (0, 2)]
pair_names = [
    "Sepal Length vs Sepal Width",
    "Petal Length vs Petal Width",
    "Sepal Length vs Petal Length"
]

for (f1, f2), pair_name in zip(feature_pairs, pair_names):
    plot_decision_boundary_2d(
        gnb_scratch, X_iris, y_iris,
        feature_indices=(f1, f2),
        feature_names=feature_names,
        class_names=target_names,
        title=f"Decision Boundary - {pair_name}"
    )

In [ ]:
def plot_class_distributions(model, X, y, feature_names=None, class_names=None):
    """
    Plot the learned Gaussian distributions for each feature and class.
    """
    n_features = X.shape[1]
    n_classes = len(model.classes_)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    
    colors = plt.cm.Set1(np.linspace(0, 1, n_classes))
    
    for feat_idx in range(min(4, n_features)):
        ax = axes[feat_idx]
        
        # Get feature range
        x_min = X[:, feat_idx].min() - 1
        x_max = X[:, feat_idx].max() + 1
        x_range = np.linspace(x_min, x_max, 200)
        
        for class_idx in range(n_classes):
            # Get learned parameters
            mean = model.theta_[class_idx, feat_idx]
            var = model.var_[class_idx, feat_idx]
            std = np.sqrt(var)
            
            # Compute Gaussian PDF
            pdf = stats.norm.pdf(x_range, mean, std)
            
            label = class_names[class_idx] if class_names else f'Class {class_idx}'
            ax.plot(x_range, pdf, color=colors[class_idx], linewidth=2, label=label)
            ax.fill_between(x_range, pdf, alpha=0.2, color=colors[class_idx])
            
            # Mark mean
            ax.axvline(mean, color=colors[class_idx], linestyle='--', alpha=0.7)
        
        # Add histograms of actual data
        for class_idx in range(n_classes):
            mask = y == model.classes_[class_idx]
            ax.hist(X[mask, feat_idx], bins=15, alpha=0.2, 
                   color=colors[class_idx], density=True)
        
        feat_name = feature_names[feat_idx] if feature_names else f'Feature {feat_idx}'
        ax.set_xlabel(feat_name, fontsize=11)
        ax.set_ylabel('Density', fontsize=11)
        ax.set_title(f'Class Distributions - {feat_name}', fontsize=12)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('Gaussian Naive Bayes - Learned Class-Conditional Distributions', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Plot class distributions
plot_class_distributions(gnb_scratch, X_iris, y_iris, 
                         feature_names=feature_names, 
                         class_names=target_names)

In [ ]:
def plot_probability_heatmap(model, X, y, sample_indices=None, 
                              class_names=None, title="Class Probabilities"):
    """
    Plot heatmap of predicted class probabilities.
    """
    if sample_indices is None:
        sample_indices = np.arange(min(20, len(X)))
    
    X_subset = X[sample_indices]
    y_subset = y[sample_indices]
    proba = model.predict_proba(X_subset)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    im = ax.imshow(proba, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    
    # Add text annotations
    for i in range(len(sample_indices)):
        for j in range(proba.shape[1]):
            text = ax.text(j, i, f'{proba[i, j]:.2f}',
                          ha='center', va='center', fontsize=9,
                          color='white' if proba[i, j] > 0.5 else 'black')
    
    # Labels
    if class_names is not None:
        ax.set_xticks(np.arange(len(class_names)))
        ax.set_xticklabels(class_names, fontsize=11)
    else:
        ax.set_xticks(np.arange(proba.shape[1]))
    
    # Y-axis: sample index with true label
    if class_names is not None:
        y_labels = [f"Sample {i} (True: {class_names[y_subset[i]]})" 
                   for i in range(len(sample_indices))]
    else:
        y_labels = [f"Sample {i} (True: {y_subset[i]})" 
                   for i in range(len(sample_indices))]
    ax.set_yticks(np.arange(len(sample_indices)))
    ax.set_yticklabels(y_labels, fontsize=9)
    
    ax.set_xlabel('Predicted Class', fontsize=12)
    ax.set_ylabel('Sample', fontsize=12)
    ax.set_title(title, fontsize=14)
    
    plt.colorbar(im, ax=ax, label='Probability')
    plt.tight_layout()
    plt.show()

# Plot probability heatmap for test samples
plot_probability_heatmap(gnb_scratch, X_test, y_test, 
                          sample_indices=np.arange(15),
                          class_names=target_names,
                          title="Class Probability Predictions - Iris Test Set")

In [ ]:
def plot_feature_importance_mnb(model, feature_names, class_names, top_n=10):
    """
    Visualize most important features for Multinomial NB.
    Based on log probability differences between classes.
    """
    n_classes = len(class_names)
    
    fig, axes = plt.subplots(1, n_classes, figsize=(5*n_classes, 6))
    
    for idx, (ax, class_name) in enumerate(zip(axes, class_names)):
        # Get log probabilities for this class
        log_probs = model.feature_log_prob_[idx]
        
        # Get top features
        top_indices = np.argsort(log_probs)[-top_n:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        top_scores = log_probs[top_indices]
        
        # Plot horizontal bar chart
        y_pos = np.arange(len(top_words))
        ax.barh(y_pos, np.exp(top_scores), color=plt.cm.Set2(idx / n_classes))
        ax.set_yticks(y_pos)
        ax.set_yticklabels(top_words, fontsize=10)
        ax.set_xlabel('P(word | class)', fontsize=11)
        ax.set_title(f'Top Words: {class_name}', fontsize=12)
        ax.invert_yaxis()
        ax.grid(True, alpha=0.3, axis='x')
    
    plt.suptitle('Most Important Words per Class (Multinomial NB)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Plot feature importance for text classification
vocab = vectorizer.get_feature_names_out()
plot_feature_importance_mnb(mnb_scratch, vocab, categories, top_n=10)

## 6. Use Cases & Guidelines <a id='use-cases'></a>

### When to Use Naive Bayes

#### Best Use Cases:

1. **Text Classification**
   - Spam detection (email filtering)
   - Sentiment analysis
   - Document categorization
   - Language detection
   - Topic classification

2. **Real-time Prediction**
   - When fast predictions are required
   - Online learning scenarios
   - Streaming data classification

3. **Multi-class Classification**
   - Natural handling of multiple classes
   - No need for one-vs-rest or one-vs-one schemes

4. **High-dimensional Sparse Data**
   - Text data with large vocabularies
   - Feature vectors with many zeros
   - Bag-of-words representations

5. **Baseline Models**
   - Quick first model to establish baseline
   - Benchmark for more complex algorithms

6. **Small Training Sets**
   - Works well with limited data
   - Less prone to overfitting than complex models

---

### When NOT to Use Naive Bayes

1. **Feature Dependencies**
   - When features are highly correlated
   - Complex feature interactions exist
   - Use: Random Forest, Gradient Boosting, Neural Networks

2. **Numeric Prediction**
   - Not suitable for regression tasks
   - Use: Linear Regression, Neural Networks

3. **Zero-frequency Problem**
   - When test data has features not seen in training
   - Mitigated by Laplace smoothing but still an issue

4. **Well-calibrated Probabilities Needed**
   - Naive Bayes tends to push probabilities to 0 or 1
   - Use: Calibrated classifiers, Logistic Regression

5. **Non-Gaussian Continuous Features**
   - Gaussian NB assumes normal distribution
   - Heavy-tailed or multimodal distributions
   - Use: Tree-based methods, KDE-based NB

---

### Data Requirements

| Variant | Feature Type | Preprocessing |
|---------|-------------|---------------|
| Gaussian NB | Continuous | StandardScaler (optional) |
| Multinomial NB | Counts/Frequencies | Non-negative, CountVectorizer/TfidfVectorizer |
| Bernoulli NB | Binary | Binarize continuous features |

---

### Pros and Cons

#### Pros:
- Fast training and prediction (O(n) complexity)
- Works well with high-dimensional data
- Handles multi-class problems naturally
- Requires minimal hyperparameter tuning
- Good with small datasets
- Interpretable (feature likelihoods)
- Robust to irrelevant features

#### Cons:
- Assumes feature independence (often violated)
- Poor probability estimates (uncalibrated)
- Zero-frequency problem without smoothing
- Limited expressiveness (linear boundaries)
- Gaussian NB sensitive to outliers

---

### Hyperparameter Tuning Guidelines

| Parameter | Range | Notes |
|-----------|-------|-------|
| alpha (smoothing) | 0.001 - 10.0 | Higher = more regularization |
| var_smoothing | 1e-9 - 1e-6 | Prevents division by zero |
| binarize threshold | 0.0 - 1.0 | For Bernoulli NB |

**Typical values:**
- alpha = 1.0 (Laplace smoothing) is a good default
- For text classification, try alpha in [0.01, 0.1, 1.0]

---

### Common Pitfalls & Solutions

1. **Zero Probability Problem**
   - Issue: Unseen feature-class combinations
   - Solution: Use Laplace smoothing (alpha > 0)

2. **Feature Scaling**
   - Gaussian NB: Scale features to similar ranges
   - Multinomial NB: Do NOT scale (need counts)

3. **Numerical Underflow**
   - Issue: Product of small probabilities -> 0
   - Solution: Use log probabilities (implemented above)

4. **Correlated Features**
   - Issue: Violates independence assumption
   - Solution: Feature selection, PCA, or different algorithm

## 7. Comparison with sklearn <a id='comparison'></a>

In [ ]:
# Compare Gaussian Naive Bayes implementations
print("="*70)
print("GAUSSIAN NAIVE BAYES COMPARISON")
print("="*70)

# Our implementation
gnb_ours = GaussianNaiveBayesScratch(var_smoothing=1e-9)
gnb_ours.fit(X_train, y_train)

# sklearn implementation
gnb_sklearn = GaussianNB(var_smoothing=1e-9)
gnb_sklearn.fit(X_train, y_train)

# Predictions
pred_ours = gnb_ours.predict(X_test)
pred_sklearn = gnb_sklearn.predict(X_test)

# Probabilities
proba_ours = gnb_ours.predict_proba(X_test)
proba_sklearn = gnb_sklearn.predict_proba(X_test)

print("\n--- Performance Metrics ---")
print(f"{'Metric':<30} {'Our Impl.':<15} {'sklearn':<15}")
print("-"*60)
print(f"{'Train Accuracy':<30} {gnb_ours.score(X_train, y_train):<15.4f} {gnb_sklearn.score(X_train, y_train):<15.4f}")
print(f"{'Test Accuracy':<30} {gnb_ours.score(X_test, y_test):<15.4f} {gnb_sklearn.score(X_test, y_test):<15.4f}")
print(f"{'Prediction Agreement':<30} {np.mean(pred_ours == pred_sklearn):<15.4f}")

print("\n--- Learned Parameters Comparison ---")
print("\nClass Priors:")
print(f"  Ours:    {gnb_ours.class_prior_.round(4)}")
print(f"  sklearn: {gnb_sklearn.class_prior_.round(4)}")
print(f"  Match:   {np.allclose(gnb_ours.class_prior_, gnb_sklearn.class_prior_)}")

print("\nClass Means (theta):")
print(f"  Ours:")
print(gnb_ours.theta_.round(4))
print(f"  sklearn:")
print(gnb_sklearn.theta_.round(4))
print(f"  Match: {np.allclose(gnb_ours.theta_, gnb_sklearn.theta_)}")

In [ ]:
# Compare Multinomial Naive Bayes implementations
print("="*70)
print("MULTINOMIAL NAIVE BAYES COMPARISON")
print("="*70)

# Our implementation
mnb_ours = MultinomialNaiveBayesScratch(alpha=1.0)
mnb_ours.fit(X_train_text, y_train_text)

# sklearn implementation
mnb_sklearn = MultinomialNB(alpha=1.0)
mnb_sklearn.fit(X_train_text, y_train_text)

# Predictions
pred_ours_mnb = mnb_ours.predict(X_test_text)
pred_sklearn_mnb = mnb_sklearn.predict(X_test_text)

# Probabilities
proba_ours_mnb = mnb_ours.predict_proba(X_test_text)
proba_sklearn_mnb = mnb_sklearn.predict_proba(X_test_text)

print("\n--- Performance Metrics ---")
print(f"{'Metric':<30} {'Our Impl.':<15} {'sklearn':<15}")
print("-"*60)
print(f"{'Train Accuracy':<30} {mnb_ours.score(X_train_text, y_train_text):<15.4f} {mnb_sklearn.score(X_train_text, y_train_text):<15.4f}")
print(f"{'Test Accuracy':<30} {mnb_ours.score(X_test_text, y_test_text):<15.4f} {mnb_sklearn.score(X_test_text, y_test_text):<15.4f}")
print(f"{'Prediction Agreement':<30} {np.mean(pred_ours_mnb == pred_sklearn_mnb):<15.4f}")

print("\n--- Probability Comparison ---")
prob_diff = np.abs(proba_ours_mnb - proba_sklearn_mnb)
print(f"Mean Absolute Probability Difference: {prob_diff.mean():.6f}")
print(f"Max Absolute Probability Difference:  {prob_diff.max():.6f}")

In [ ]:
# Compare Bernoulli Naive Bayes implementations
print("="*70)
print("BERNOULLI NAIVE BAYES COMPARISON")
print("="*70)

# Our implementation
bnb_ours = BernoulliNaiveBayesScratch(alpha=1.0, binarize=0.0)
bnb_ours.fit(X_train_text, y_train_text)

# sklearn implementation
bnb_sklearn = BernoulliNB(alpha=1.0, binarize=0.0)
bnb_sklearn.fit(X_train_text, y_train_text)

# Predictions
pred_ours_bnb = bnb_ours.predict(X_test_text)
pred_sklearn_bnb = bnb_sklearn.predict(X_test_text)

# Probabilities
proba_ours_bnb = bnb_ours.predict_proba(X_test_text)
proba_sklearn_bnb = bnb_sklearn.predict_proba(X_test_text)

print("\n--- Performance Metrics ---")
print(f"{'Metric':<30} {'Our Impl.':<15} {'sklearn':<15}")
print("-"*60)
print(f"{'Train Accuracy':<30} {bnb_ours.score(X_train_text, y_train_text):<15.4f} {bnb_sklearn.score(X_train_text, y_train_text):<15.4f}")
print(f"{'Test Accuracy':<30} {bnb_ours.score(X_test_text, y_test_text):<15.4f} {bnb_sklearn.score(X_test_text, y_test_text):<15.4f}")
print(f"{'Prediction Agreement':<30} {np.mean(pred_ours_bnb == pred_sklearn_bnb):<15.4f}")

print("\n--- Probability Comparison ---")
prob_diff_bnb = np.abs(proba_ours_bnb - proba_sklearn_bnb)
print(f"Mean Absolute Probability Difference: {prob_diff_bnb.mean():.6f}")
print(f"Max Absolute Probability Difference:  {prob_diff_bnb.max():.6f}")

In [ ]:
# Visualize probability comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Gaussian NB
ax = axes[0]
for i in range(3):
    ax.scatter(proba_sklearn[:, i], proba_ours[:, i], alpha=0.5, label=f'Class {i}')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Agreement')
ax.set_xlabel('sklearn Probabilities', fontsize=11)
ax.set_ylabel('Our Implementation', fontsize=11)
ax.set_title('Gaussian NB - Probability Comparison', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Multinomial NB
ax = axes[1]
for i in range(3):
    ax.scatter(proba_sklearn_mnb[:, i], proba_ours_mnb[:, i], alpha=0.5, label=f'Class {i}')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Agreement')
ax.set_xlabel('sklearn Probabilities', fontsize=11)
ax.set_ylabel('Our Implementation', fontsize=11)
ax.set_title('Multinomial NB - Probability Comparison', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Bernoulli NB
ax = axes[2]
for i in range(3):
    ax.scatter(proba_sklearn_bnb[:, i], proba_ours_bnb[:, i], alpha=0.5, label=f'Class {i}')
ax.plot([0, 1], [0, 1], 'k--', label='Perfect Agreement')
ax.set_xlabel('sklearn Probabilities', fontsize=11)
ax.set_ylabel('Our Implementation', fontsize=11)
ax.set_title('Bernoulli NB - Probability Comparison', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Timing comparison
import time

print("="*70)
print("TIMING COMPARISON")
print("="*70)

# Generate larger dataset for timing
from sklearn.datasets import make_classification
X_large, y_large = make_classification(n_samples=10000, n_features=100, 
                                        n_informative=50, n_classes=5,
                                        random_state=42)

# Gaussian NB timing
print("\nGaussian Naive Bayes (10,000 samples, 100 features, 5 classes):")

# Our implementation
start = time.time()
gnb_ours_large = GaussianNaiveBayesScratch()
gnb_ours_large.fit(X_large, y_large)
_ = gnb_ours_large.predict(X_large)
time_ours = time.time() - start

# sklearn
start = time.time()
gnb_sklearn_large = GaussianNB()
gnb_sklearn_large.fit(X_large, y_large)
_ = gnb_sklearn_large.predict(X_large)
time_sklearn = time.time() - start

print(f"  Our Implementation: {time_ours:.4f} seconds")
print(f"  sklearn:            {time_sklearn:.4f} seconds")
print(f"  Ratio (ours/sklearn): {time_ours/time_sklearn:.2f}x")

## Summary & Key Takeaways

### What We Learned

1. **Mathematical Foundation**: Bayes' theorem, prior, likelihood, posterior, and the naive independence assumption

2. **Three Variants Implemented**:
   - **Gaussian NB**: For continuous features (assumes normal distribution)
   - **Multinomial NB**: For count data (word frequencies, TF-IDF)
   - **Bernoulli NB**: For binary features (word presence/absence)

3. **Key Implementation Details**:
   - Log probabilities prevent numerical underflow
   - Laplace smoothing handles zero-frequency problem
   - Variance smoothing ensures numerical stability

4. **Practical Insights**:
   - Extremely fast training and prediction
   - Works surprisingly well despite violated assumptions
   - Excellent for text classification and high-dimensional data

### Key Observations from sklearn Comparison

- Our implementations match sklearn's results closely
- Probability estimates are nearly identical
- sklearn is more optimized for speed but our implementation is educational

### Next Steps

1. **Explore Variants**: Complement NB, Semi-supervised NB
2. **Probability Calibration**: Platt scaling, isotonic regression
3. **Feature Engineering**: TF-IDF weighting, n-grams for text
4. **Ensemble Methods**: Combine NB with other classifiers
5. **Online Learning**: Incremental updates for streaming data